### 中文抽取式问答

手动下载模型文件并加载
访问 Hugging Face 模型页面
打开浏览器，访问 Hugging Face 模型库网站：https://huggingface.co/distilbert/distilbert-base-multilingual-cased


下载模型文件
从模型页面下载以下文件：

config.json
pytorch_model.bin
vocab.txt 或 tokenizer.json
保存文件到本地目录
将下载的文件保存到一个本地目录，例如 ./distilbert-base-multilingual-cased/。

加载本地模型
使用以下代码加载本地模型：

In [5]:
from transformers import DistilBertTokenizer, DistilBertModel

model_path = './distilbert-base-multilingual-cased'

tokenizer = DistilBertTokenizer.from_pretrained(model_path)
model = DistilBertModel.from_pretrained(model_path)

text = "Hello, how are you?"
'''
Token        ID
----------------
[PAD]         0  填充字符
[UNK]       100  未知字符
[CLS]       101  句子开头
[SEP]       102  句子结束
'''
inputs = tokenizer(
    text, return_tensors="pt")  # 'pt'`: Return PyTorch `torch.Tensor` objects.
# 解包参数等价于outputs = model(input_ids=inputs["input_ids"],attention_mask=inputs["attention_mask"])
outputs = model(**inputs)
print("inputs", inputs)
print("outputs", outputs)
print(outputs.last_hidden_state.size())  # Bert encoder最后一层的输出文本表征维度为(batch_size, sequence_length, hidden_size)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 24578.40it/s]
[transformers] DistilBertModel LOAD REPORT from: ./distilbert-base-multilingual-cased
Key               | Status     |  | 
------------------+------------+--+-
qa_outputs.weight | UNEXPECTED |  | 
qa_outputs.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


inputs {'input_ids': tensor([[ 101, 8667,  117, 1293, 1132, 1128,  136,  102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}
outputs BaseModelOutput(last_hidden_state=tensor([[[ 0.5033, -0.1641,  0.6830,  ..., -0.7116,  0.1367, -0.3267],
         [ 0.0480, -0.2771,  2.0183,  ..., -0.9260,  0.1186, -0.3417],
         [ 0.8671,  0.4700,  1.7358,  ..., -0.6167, -0.5366, -0.1597],
         ...,
         [ 0.8481, -0.4150,  1.2586,  ..., -0.8394, -0.2204, -0.0782],
         [ 0.7422, -0.3642,  1.3790,  ..., -0.6849, -0.2361,  0.2979],
         [ 0.9348, -0.7310,  0.6012,  ..., -0.4271,  0.6050, -0.2945]]],
       grad_fn=<NativeLayerNormBackward0>), hidden_states=None, attentions=None)
torch.Size([1, 8, 768])


## Transformer英文问答练习

In [6]:
from transformers import pipeline

### Construct Q&A task

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_path = "./distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForQuestionAnswering.from_pretrained(model_path)

question = "What are the learning models?"

context = "Machine learning models are something, which are based on a combination \
    of several key components, furthermore there are concepts, \
    which enable them to learn patterns from data and make predictions or decisions."


inputs = tokenizer(question, context, return_tensors="pt")  # 'pt'`: Return PyTorch `torch.Tensor` objects.

with torch.no_grad():
    outputs = model(**inputs)

start = torch.argmax(outputs.start_logits)  # 返回最可能的 start token 的索引
end = torch.argmax(outputs.end_logits)    # 返回最可能的 end token 的索引

print("Start:", start.item())
print("End:", end.item())

answer_tokens = inputs["input_ids"][0][start:end + 1]
print("Answer tokens:", answer_tokens) # token id编码包括question和context两个部分
answer = tokenizer.decode(
    answer_tokens,  # `answer_tokens`: 预测的 token id 列表或 1D tensor（可用 .tolist() 转为 list）
    skip_special_tokens=True  # 跳过特殊 tokens（如 [CLS], [SEP], [PAD]），返回更干净的自然语言字符串
)

start_score = torch.softmax(outputs.start_logits[0], dim=0)[start]
end_score = torch.softmax(outputs.end_logits[0], dim=0)[end]

score = (start_score * end_score).item()

print("Inputs:", inputs)
print("Answer:", answer)
print("Score:", score)  # 通过将预测答案起始和结束位置的概率相乘来得到模型置信度得分。

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 17158.75it/s]


Start: 16
End: 23
Answer tokens: tensor([1359, 1113,  170, 4612, 1104, 1317, 2501, 5644])
Inputs: {'input_ids': tensor([[  101,  1327,  1132,  1103,  3776,  3584,   136,   102,  7792,  3776,
          3584,  1132,  1380,   117,  1134,  1132,  1359,  1113,   170,  4612,
          1104,  1317,  2501,  5644,   117,  1748,  4982,  1175,  1132,  8550,
           117,  1134,  9396,  1172,  1106,  3858,  6692,  1121,  2233,  1105,
          1294, 23770,  1137,  6134,   119,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Answer: based on a combination of several key components
Score: 0.3561100661754608


### Transformer中文问答练习

In [9]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_path = "./bert-multi-cased-finetuned-xquadv1"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForQuestionAnswering.from_pretrained(model_path)

question = "机器学习是什么的分支"
context = "机器学习是人工智能的一个分支，是个很热门的专业。"

inputs = tokenizer(question, context, return_tensors="pt")

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

start_logits = outputs.start_logits[0]   # 答案起点得分
end_logits = outputs.end_logits[0]       # 答案终点得分

start_idx = start_logits.argmax().item()
end_idx = (start_idx + end_logits[start_idx:].argmax()).item()

answer = tokenizer.decode(
    inputs["input_ids"][0][start_idx:end_idx + 1],
    skip_special_tokens=True,
)

score = (start_logits.softmax(-1)[start_idx] * end_logits.softmax(-1)[end_idx]).item()
print("Result:", {"answer": answer, "score": score})
print("Answer:", answer)
print("Score:", score)


[transformers] The tokenizer you are loading from './bert-multi-cased-finetuned-xquadv1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25429.32it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: ./bert-multi-cased-finetuned-xquadv1
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Result: {'answer': '人 工 智 能', 'score': 0.9577576518058777}
Answer: 人 工 智 能
Score: 0.9577576518058777
